# Neural Network: Reading Handwritten Digits

This is the classic real-world use of neural networks: **reading handwriting**. It's the same idea banks use
to read cheque amounts and post offices use to read ZIP codes.

**The data is real.** 1,797 digits (0–9) handwritten by real people on paper forms, scanned and shrunk to
8×8 pixel images. It comes from the UCI Machine Learning Repository and ships bundled with scikit-learn,
so there's nothing to download.

Requirements: `pip install scikit-learn numpy`

In [1]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings("ignore")

## Step 1: Load the real data

Each image is 8×8 = **64 pixels**. Each pixel is a number from 0 (white paper) to 16 (dark ink).
So to the network, every handwritten digit is just a list of 64 numbers, plus the correct answer (the label).

In [2]:
digits = load_digits()
X = digits.data      # 1797 images, each a row of 64 pixel values
y = digits.target    # the correct digit for each image (0-9)

print("Images:", X.shape[0])
print("Pixels per image:", X.shape[1])
print("Digits:", sorted(set(y.tolist())))
print("\nThe first image as raw numbers (what the network actually sees):")
print(digits.images[0].astype(int))
print("Label:", y[0])

Images: 1797
Pixels per image: 64
Digits: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

The first image as raw numbers (what the network actually sees):
[[ 0  0  5 13  9  1  0  0]
 [ 0  0 13 15 10 15  5  0]
 [ 0  3 15  2  0 11  8  0]
 [ 0  4 12  0  0  8  8  0]
 [ 0  5  8  0  0  9  8  0]
 [ 0  4 11  0  1 12  7  0]
 [ 0  2 14  5 10 12  0  0]
 [ 0  0  6 13 10  0  0  0]]
Label: 0


## Step 2: Look at the digits

Raw numbers are hard for humans to read, so here's a helper that prints an image as text:
dark pixels become `#`, medium ones `+`, light ones `.`.

In [3]:
def show(pixels):
    img = np.array(pixels).reshape(8, 8)
    for row in img:
        print("  " + "".join("##" if v > 10 else "++" if v > 4 else ".." if v > 0 else "  " for v in row))

for i in [0, 1, 2, 7]:
    print(f"Label: {y[i]}")
    show(X[i])
    print()

Label: 0
      ++##++..    
      ####++##++  
    ..##..  ##++  
    ..##    ++++  
    ++++    ++++  
    ..##  ..##++  
    ..##++++##    
      ++##++      

Label: 1
        ####++    
        ####++    
      ..####++    
    ++######..    
      ..####..    
      ..####++    
      ..####++    
        ####++    

Label: 2
        ..####    
      ..######    
      ++##++##    
      ..++####    
    ..++####..    
    ++####++      
    ..########++  
        ..####++  

Label: 7
      ++++######..
      ++++..####  
          ++##..  
    ..++++####++  
    ..######..    
        ##++      
      ++##..      
      ##++        



## Step 3: Split into training and test sets

- **Training set (80%):** what the network studies from
- **Test set (20%):** handwriting it never sees during training, our "real exam"

We also scale pixels from 0–16 down to 0–1. Neural networks learn more smoothly with small numbers.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X / 16.0, y, test_size=0.2, random_state=42, stratify=y
)
print("Training images:", len(X_train))
print("Test images:    ", len(X_test))

Training images: 1437
Test images:     360


## Step 4: Build the network

- **Input layer:** 64 neurons, one per pixel
- **Hidden layer:** 32 neurons with ReLU. These learn to detect strokes, loops and curves.
- **Output layer:** 10 neurons, one per digit. Softmax turns them into probabilities that add up to 100%.

The prediction is whichever of the 10 output neurons is most confident.

In [5]:
batch_size = 32
model = MLPClassifier(
    hidden_layer_sizes=(32,),   # one hidden layer with 32 neurons
    activation="relu",
    solver="adam",              #  (Adaptive Moment Estimation) is a popular algorithm used in machine learning to train neural networks faster and more accurately
    batch_size=batch_size,
    learning_rate_init=0.001,
    random_state=42,
)

iterations_per_epoch = int(np.ceil(len(X_train) / batch_size))
learnable = 64 * 32 + 32 + 32 * 10 + 10
print(f"Learnable weights + biases: {learnable}")
print(f"{len(X_train)} training images / batch of {batch_size} = {iterations_per_epoch} iterations per epoch")

Learnable weights + biases: 2410
1437 training images / batch of 32 = 45 iterations per epoch


## Step 5: Train it, one epoch at a time

`partial_fit` runs **exactly one epoch**: it goes through every training image once, in batches of 32,
doing forward pass → loss → backprop → weight update for each batch.

After each epoch we check accuracy on both the training images and the test images it has never seen.

In [6]:
epochs = 25
iteration = 0

print("Epoch | Iterations | Loss   | Train acc | Test acc")
print("------+------------+--------+-----------+---------")
for epoch in range(1, epochs + 1):
    model.partial_fit(X_train, y_train, classes=np.arange(10))
    iteration += iterations_per_epoch
    train_acc = model.score(X_train, y_train) * 100
    test_acc = model.score(X_test, y_test) * 100
    if epoch <= 5 or epoch % 5 == 0:
        print(f"  {epoch:2d}  |    {iteration:5d}   | {model.loss_:.4f} |  {train_acc:5.1f}%   |  {test_acc:5.1f}%")

Epoch | Iterations | Loss   | Train acc | Test acc
------+------------+--------+-----------+---------
   1  |       45   | 2.1460 |   43.4%   |   42.8%
   2  |       90   | 1.7363 |   64.9%   |   63.6%
   3  |      135   | 1.3418 |   80.7%   |   80.3%
   4  |      180   | 1.0203 |   86.4%   |   85.8%
   5  |      225   | 0.7837 |   89.3%   |   88.3%
  10  |      450   | 0.3485 |   93.7%   |   91.4%
  15  |      675   | 0.2245 |   95.6%   |   93.3%
  20  |      900   | 0.1653 |   97.1%   |   95.3%
  25  |     1125   | 0.1295 |   97.7%   |   95.6%


Watch the first few epochs: the network goes from guessing to recognising most digits very quickly,
then slowly polishes. Test accuracy stays close to training accuracy, so it's genuinely **learning
what digits look like**, not memorising these particular images.


## Step 6: Watch it read some test digits

For each digit, the network outputs a probability for all 10 possibilities. We show its top 3 guesses.

In [7]:
probs = model.predict_proba(X_test)
for i in [0, 5, 11]:
    show(X_test[i] * 16)
    top3 = np.argsort(probs[i])[::-1][:3]
    guesses = ", ".join(f"{d} ({probs[i][d]*100:.1f}%)" for d in top3)
    print(f"Real: {y_test[i]}  |  Network's top guesses: {guesses}\n")

      ++######..  
      ##++  ..    
      ##......    
    ..########..  
      ..    ++++  
    ..      ##++  
    ++##....##..  
      ++####..    
Real: 5  |  Network's top guesses: 5 (89.1%), 9 (8.6%), 0 (1.2%)

    ..######..    
    ########..    
    ......##..    
        ++##      
      ..##++      
      ####..##++  
      ########++  
      ######++    
Real: 2  |  Network's top guesses: 2 (56.5%), 3 (41.1%), 8 (1.9%)

      ++######    
      ####++++    
    ..##++        
    ++########    
    ..########++  
      ......##++  
      ..++####    
      ####++..    
Real: 5  |  Network's top guesses: 5 (99.6%), 9 (0.2%), 3 (0.1%)



## Step 7: Draw your own digit

Edit the 8×8 grid below and re-run the cell. Use `#` for ink and `.` for empty paper.
Every row must have exactly 8 characters.

In [8]:
my_drawing = [
    "..####..",
    ".#....#.",
    "......#.",
    ".....#..",
    "....#...",
    "...#....",
    "..#.....",
    ".######.",
]

pixels = np.array([[1.0 if ch == "#" else 0.0 for ch in row] for row in my_drawing]).reshape(1, -1)
p = model.predict_proba(pixels)[0]
top3 = np.argsort(p)[::-1][:3]
print("The network thinks you drew a", top3[0])
print("Top guesses:", ", ".join(f"{d} ({p[d]*100:.1f}%)" for d in top3))

The network thinks you drew a 2
Top guesses: 2 (97.4%), 3 (2.4%), 8 (0.1%)
